# 01. Extracción de las fuentes de datos

Proyecto final del Máster en Data Analytics.

Este cuaderno documenta de dónde salen los datos y qué contiene cada fuente antes
de tocar nada.

## La pregunta de partida

¿Cómo ha crecido el powerlifting femenino en el mundo, qué determina el rendimiento
de una atleta, y qué papel juega el contexto social de su país?

## Las fuentes

El requisito es partir de dos conjuntos de datos de fuentes distintas y unirlos.
Aquí se combinan tres orígenes independientes, agrupados en dos bloques temáticos:

| Bloque | Origen | Aportación | Acceso |
|---|---|---|---|
| Fuente 1 | [OpenPowerlifting](https://www.openpowerlifting.org) | Resultados reales de competición en todo el mundo | Volcado CSV, dominio público |
| Fuente 2a | [Banco Mundial](https://data.worldbank.org) | Indicadores económicos y de participación femenina | API REST pública, sin clave |
| Fuente 2b | [PNUD](https://hdr.undp.org) | IDH e Índice de Desigualdad de Género | CSV público |

El nexo entre ambos bloques es el país, mediante código ISO3, y el año.

In [1]:
import sys
from pathlib import Path

# Permite importar los modulos del proyecto desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import numpy as np
from IPython.display import Image, display

import config as cfg
import estilo

estilo.aplicar()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
print("Entorno listo. Raiz del proyecto:", RAIZ.name)

Entorno listo. Raiz del proyecto: Proyecto-final


## Fuente 1. OpenPowerlifting

Archivo histórico y abierto de resultados de competiciones de powerlifting. Los
datos están en dominio público: la propia licencia renuncia a los derechos sobre el
conjunto y solo pide atribución de forma voluntaria.

El volcado completo son unos 160 MB comprimidos y 4.001.901 registros de ambos
sexos. Como el objeto de estudio es el powerlifting femenino, el script de
extracción filtra el subconjunto de mujeres y lo guarda comprimido.

In [2]:
f1 = pd.read_csv(cfg.F1_RAW, low_memory=False)
print(f"Registros femeninos en bruto: {f1.shape[0]:,} filas x {f1.shape[1]} columnas")
print(f"Periodo: {f1['Date'].min()} a {f1['Date'].max()}")
f1.head(4)

Registros femeninos en bruto: 1,120,543 filas x 42 columnas
Periodo: 1975-09-05 a 2026-08-09


,Name,Sex,Event,Equipment,Age,AgeClass,BirthYearClass,Division,BodyweightKg,WeightClassKg,Squat1Kg,Squat2Kg,Squat3Kg,Squat4Kg,Best3SquatKg,Bench1Kg,Bench2Kg,Bench3Kg,Bench4Kg,Best3BenchKg,Deadlift1Kg,Deadlift2Kg,Deadlift3Kg,Deadlift4Kg,Best3DeadliftKg,TotalKg,Place,Dots,Wilks,Glossbrenner,Goodlift,Tested,Country,State,Federation,ParentFederation,Date,MeetCountry,MeetState,MeetTown,MeetName,Sanctioned
0,E.S. Denisenko,F,B,Raw,28.5,24-34,24-39,Open,67.3,NaN,NaN,NaN,NaN,NaN,NaN,-40.0,-45.0,-45.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DQ,NaN,NaN,NaN,NaN,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
1,I.S. Lebetskaya,F,B,Raw,43.5,40-44,40-49,Open,73.2,NaN,NaN,NaN,NaN,NaN,NaN,80.0,85.0,90.0,NaN,90.0,NaN,NaN,NaN,NaN,NaN,90.0,1,88.80,86.89,76.50,70.06,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
2,K. Yakimovich,F,B,Raw,26.5,24-34,24-39,Open,60.6,NaN,NaN,NaN,NaN,NaN,NaN,40.0,42.5,45.0,NaN,45.0,NaN,NaN,NaN,NaN,NaN,45.0,2,49.57,49.79,43.98,38.42,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
3,A.G. Golneva,F,B,Raw,19.5,20-23,19-23,Juniors 17-21,50.3,NaN,NaN,NaN,NaN,NaN,NaN,32.5,35.0,-37.5,NaN,35.0,NaN,NaN,NaN,NaN,NaN,35.0,2,43.67,44.76,39.73,34.55,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes


### Qué significa cada columna relevante

Vale detenerse aquí, porque hay dos convenciones del formato que, si se pasan por
alto, arruinan el análisis.

La columna `Event` indica la modalidad. `SBD` es powerlifting completo (squat,
bench, deadlift) y es la única que permite comparar totales entre atletas. `B` es
solo press de banca, `D` solo peso muerto, y así con el resto.

Las columnas `Best3SquatKg`, `Best3BenchKg` y `Best3DeadliftKg` recogen el mejor
levantamiento válido de cada movimiento. Aquí está la trampa: un valor negativo no
es un error, significa intento fallado. Tratarlo como un número sin más produciría
totales absurdos.

`Equipment` indica el material permitido, desde `Raw` (sin apoyo) hasta `Multi-ply`
(varias capas de material elástico).

`Dots`, `Wilks` y `Goodlift` son índices que normalizan la marca según el peso
corporal. Usan coeficientes distintos para hombres y para mujeres, así que sirven
para comparar dentro de cada sexo pero no entre sexos.

`Place` recoge el puesto final, o un código como `DQ` (descalificada) o `NS` (no se
presentó).

In [3]:
print("MODALIDADES (Event):")
print(f1["Event"].value_counts().to_string())

print("\nEQUIPAMIENTO:")
print(f1["Equipment"].value_counts().to_string())

print("\nValores negativos = intentos fallados:")
for c in ["Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg"]:
    print(f"  {c:<18} {(f1[c] < 0).sum():>7,} negativos | {f1[c].isna().sum():>7,} nulos")

MODALIDADES (Event):
Event
SBD    895088
B      146828
D       53061
BD      19721
S        4434
SB        787
SD        624

EQUIPAMIENTO:
Equipment
Raw           550119
Single-ply    331066
Unlimited     161660
Wraps          61774
Multi-ply      15919
Straps             5

Valores negativos = intentos fallados:
  Best3SquatKg           830 negativos | 257,269 nulos
  Best3BenchKg           647 negativos | 111,433 nulos
  Best3DeadliftKg        210 negativos | 204,209 nulos


In [4]:
# Calidad de partida: cuanto falta en cada columna
nulos = (f1.isna().mean() * 100).round(1).sort_values(ascending=False)
print("Columnas con mas del 10% de valores ausentes:")
print(nulos[nulos > 10].to_string())

Columnas con mas del 10% de valores ausentes:
Squat4Kg            99.6
Bench4Kg            99.3
Deadlift4Kg         98.9
State               75.6
Squat3Kg            64.3
Squat2Kg            63.9
Squat1Kg            63.7
Deadlift3Kg         60.9
Deadlift2Kg         60.2
Deadlift1Kg         59.8
Bench3Kg            54.1
Bench2Kg            53.4
Bench1Kg            53.1
ParentFederation    45.5
Age                 38.4
BirthYearClass      36.6
AgeClass            26.8
MeetState           25.5
Best3SquatKg        23.0
Best3DeadliftKg     18.2
Tested              16.5
MeetTown            15.1
Goodlift            13.6


Los cuatro campos `*4Kg` están vacíos en más del 98% de los casos: corresponden al
cuarto intento, que solo existe en tentativas de récord. Los intentos individuales
faltan en más de la mitad de los registros, porque muchas federaciones solo
reportan el mejor levantamiento. La edad falta en un 38%, un dato importante que
condicionará el análisis por edades más adelante.

## Fuente 2a. Banco Mundial

Se descargan siete indicadores mediante la API pública, sin necesidad de clave. La
elección no es arbitraria: cada uno cubre una hipótesis sobre por qué en unos países
compiten más mujeres que en otros.

In [5]:
for codigo, nombre in cfg.INDICADORES_WB.items():
    print(f"  {codigo:<22} -> {nombre}")

wb = pd.read_csv(cfg.F2_WB_RAW)
print(f"\nFuente 2a: {wb.shape[0]:,} filas x {wb.shape[1]} columnas")
print(f"Paises/agregados: {wb['iso3'].nunique()} | Anios: {wb['anio'].min()}-{wb['anio'].max()}")
wb.head(4)

  NY.GDP.PCAP.PP.KD      -> pib_per_capita_ppa
  SP.POP.TOTL            -> poblacion_total
  SL.TLF.CACT.FE.ZS      -> tasa_actividad_femenina
  SE.TER.ENRR.FE         -> matricula_superior_femenina
  SP.DYN.LE00.FE.IN      -> esperanza_vida_femenina
  SH.XPD.CHEX.PC.CD      -> gasto_sanitario_pc
  SP.URB.TOTL.IN.ZS      -> poblacion_urbana_pct

Fuente 2a: 9,100 filas x 10 columnas
Paises/agregados: 260 | Anios: 1990-2024


,iso3,pais_wb,anio,pib_per_capita_ppa,poblacion_total,tasa_actividad_femenina,matricula_superior_femenina,esperanza_vida_femenina,gasto_sanitario_pc,poblacion_urbana_pct
0,AFE,Africa Eastern and Southern,2024,4099.889812,769280888,63.643292,NaN,68.129076,NaN,38.241441
1,AFE,Africa Eastern and Southern,2023,4076.224898,750491370,63.592598,NaN,67.914632,88.485585,37.772301
2,AFE,Africa Eastern and Southern,2022,4098.126192,731821393,59.831305,NaN,67.231203,92.035288,37.360578
3,AFE,Africa Eastern and Southern,2021,4049.969513,713090928,59.454457,8.02047,65.652968,92.350333,36.908543


## Fuente 2b. PNUD (Human Development Report)

Aporta los índices compuestos de desarrollo humano y desigualdad de género. El
fichero original viene en formato ancho, con una columna por indicador y año
(`gii_1990`, `gii_1991` y así sucesivamente), más de mil columnas en total. El
script lo convierte a formato largo.

Un detalle que costó depurar: filtrar las columnas con `startswith("hdi_")` captura
también `hdi_f_1990` y `hdi_m_1990`, lo que generaba años duplicados por país y
rompía la unión. La solución fue exigir coincidencia exacta con una expresión
regular `^<indicador>_<año>$`.

In [6]:
undp = pd.read_csv(cfg.F2_UNDP_RAW)
print(f"Fuente 2b: {undp.shape[0]:,} filas x {undp.shape[1]} columnas")
print(f"Paises: {undp['iso3'].nunique()} | Anios: {undp['anio'].min()}-{undp['anio'].max()}")
print()
for pre, nombre in cfg.INDICADORES_UNDP.items():
    print(f"  {pre:<10} -> {nombre}")
undp.head(4)

Fuente 2b: 6,435 filas x 12 columnas
Paises: 195 | Anios: 1990-2022

  hdi        -> idh
  hdi_f      -> idh_femenino
  hdi_m      -> idh_masculino
  gii        -> indice_desigualdad_gen
  gdi        -> indice_desarrollo_gen
  le_f       -> esperanza_vida_f
  eys_f      -> anios_escolar_esp_f
  mys_f      -> anios_escolar_medios_f
  gni_pc_f   -> ingreso_nacional_pc_f


,iso3,pais_undp,anio,idh,idh_femenino,idh_masculino,indice_desigualdad_gen,indice_desarrollo_gen,esperanza_vida_f,anios_escolar_esp_f,anios_escolar_medios_f,ingreso_nacional_pc_f
0,AFG,Afghanistan,1990,0.284,NaN,NaN,NaN,NaN,48.397,2.117230,0.201659,NaN
1,ALB,Albania,1990,0.649,0.625742,0.668087,NaN,0.937,76.433,11.152520,6.692814,3918.806663
2,DZA,Algeria,1990,0.593,NaN,NaN,NaN,NaN,68.517,NaN,3.091533,NaN
3,AND,Andorra,1990,NaN,NaN,NaN,NaN,NaN,82.676,9.042792,NaN,NaN


### Cómo se interpreta el Índice de Desigualdad de Género

Es la variable central de la tercera pregunta, y su escala va al revés de lo
intuitivo: va de 0 a 1, donde 0 es igualdad plena y los valores altos indican más
desigualdad. Combina salud reproductiva, empoderamiento (representación
parlamentaria y educación) y participación en el mercado laboral.

In [7]:
ejemplo = undp[(undp["anio"] == 2022) & undp["indice_desigualdad_gen"].notna()]
print("Menor desigualdad en 2022:")
print(ejemplo.nsmallest(5, "indice_desigualdad_gen")[
    ["pais_undp", "indice_desigualdad_gen", "idh"]].to_string(index=False))
print("\nMayor desigualdad en 2022:")
print(ejemplo.nlargest(5, "indice_desigualdad_gen")[
    ["pais_undp", "indice_desigualdad_gen", "idh"]].to_string(index=False))

Menor desigualdad en 2022:
  pais_undp  indice_desigualdad_gen   idh
    Denmark                   0.009 0.952
     Norway                   0.012 0.966
Switzerland                   0.018 0.967
     Sweden                   0.023 0.952
Netherlands                   0.025 0.946

Mayor desigualdad en 2022:
  pais_undp  indice_desigualdad_gen   idh
      Yemen                   0.820 0.424
    Nigeria                   0.677 0.548
    Somalia                   0.674 0.380
       Chad                   0.671 0.394
Afghanistan                   0.665 0.462


## Tablas auxiliares derivadas

Además de las fuentes, la extracción genera dos agregados de apoyo.

El primero, `participacion_por_pais_anio.csv`, cuenta hombres y mujeres por país y
año para obtener la cuota de participación femenina, que es la métrica central de la
tercera pregunta. Se calcula sobre el volcado completo y no solo sobre el subconjunto
femenino, porque necesita el denominador masculino.

El segundo, `referencia_masculina_agregada.csv`, recoge medias masculinas por año y
equipamiento, para poder situar el rendimiento femenino en contexto.

In [8]:
part = pd.read_csv(cfg.EXTERNAL / "participacion_por_pais_anio.csv")
print(f"Participacion por pais-anio: {len(part):,} combinaciones")
print(part.nlargest(6, "n_total_pais_anio").to_string(index=False))

Participacion por pais-anio: 2,039 combinaciones
 anio MeetCountry  n_mujeres_pais_anio  n_hombres_pais_anio  n_total_pais_anio  pct_participacion_femenina
 2024         USA                66903                99832             166735                       40.13
 2023         USA                63201               100212             163413                       38.68
 2025         USA                61177                90594             151771                       40.31
 2022         USA                56800                89577             146377                       38.80
 2019         USA                47114                78529             125643                       37.50
 2026         USA                53222                70943             124165                       42.86


## Un obstáculo técnico

Todas las descargas HTTPS desde Python fallaban en el equipo de trabajo con
`CERTIFICATE_VERIFY_FAILED`, mientras que `curl` funcionaba sin problema. La causa
era un antivirus que inspecciona el tráfico TLS y presenta su propia autoridad
certificadora, desconocida para el almacén de certificados de Python.

Instalar `certifi` no lo resuelve, porque el problema no es que falte una CA pública
sino que la CA interceptora es local. La solución fue el paquete `truststore`, que
delega la validación al almacén del sistema operativo:

```python
import truststore
truststore.inject_into_ssl()
```

## Resumen

| Fuente | Filas | Columnas |
|---|---|---|
| 1. OpenPowerlifting (femenino) | 1.120.543 | 42 |
| 2a. Banco Mundial | 9.100 | 10 |
| 2b. PNUD | 6.435 | 12 |

El siguiente cuaderno, `02_limpieza_transformacion.ipynb`, limpia los datos y une
las fuentes.